# 03_word2vec: Word Representations with Gutenberg's Alice in Wonderland
    
This notebook trains Continuous Bag-of-Words (CBOW) and Skip-gram Word2Vec embedding models using Gensim on sentences extracted from Project Gutenberg's *Alice in Wonderland*.


## 1. Ingest and Clean Sentences

In [1]:
import re
import nltk
from gensim.models import Word2Vec

# Load Alice in Wonderland from NLTK gutenberg corpus
nltk.download('gutenberg', quiet=True)
from nltk.corpus import gutenberg
sentences_raw = gutenberg.sents('carroll-alice.txt')

# Clean and filter tokens
cleaned_sentences = []
for s in sentences_raw:
    words = [w.lower() for w in s if re.match(r"^\w+$", w)]
    if 5 < len(words) < 30:
        cleaned_sentences.append(words)

# Select first 600 sentences for fast training
train_sentences = cleaned_sentences[:600]
print(f"Extracted {len(train_sentences)} sentences from Alice in Wonderland.")
print("Sample sentence snippet:", train_sentences[10])


Extracted 600 sentences from Alice in Wonderland.
Sample sentence snippet: ['i', 'wonder', 'if', 'i', 'shall', 'fall', 'right', 'through', 'the', 'earth']


### Output Explanation: Data Preparation
- **Corpus**: We loaded the raw sentences of Lewis Carroll's *Alice in Wonderland* locally via NLTK's reader.
- **Filtering**: We sliced sentences with a length between 5 and 30 words to feed high-quality training structures into the embedding algorithm.


## 2. Train CBOW and Skip-gram Models

In [2]:
# Train CBOW (sg=0)
cbow_model = Word2Vec(sentences=train_sentences, vector_size=20, window=3, min_count=2, sg=0, epochs=100)

# Train Skip-gram (sg=1)
sg_model = Word2Vec(sentences=train_sentences, vector_size=20, window=3, min_count=2, sg=1, epochs=100)

print("Vocab size trained:", len(cbow_model.wv.key_to_index))


Vocab size trained: 667


### Output Explanation: Training Embeddings
- **Models**: We trained a **CBOW** model and a **Skip-gram** model.
- **Parameters**: `vector_size=20` sets the embedding dimensionality, and `epochs=100` allows convergence on this small dataset.


## 3. Embedding Vector Similarities

In [3]:
print("=== CBOW Embedding vector for 'alice' ===\n", cbow_model.wv["alice"])

cbow_sim = cbow_model.wv.similarity("alice", "rabbit")
sg_sim = sg_model.wv.similarity("alice", "rabbit")
print(f"\nCosine Similarity ('alice' vs 'rabbit'):")
print(f"  CBOW Similarity: {cbow_sim:.4f}")
print(f"  Skip-gram Similarity: {sg_sim:.4f}")


=== CBOW Embedding vector for 'alice' ===
 [-2.0215733   0.4512715   0.71786785 -0.6574325   0.16905674  0.18413292
  0.889511   -0.65519303 -0.01113038  0.08639467  0.05543453 -2.1029193
  0.8108308  -0.1011164   0.16339599  0.2838335   0.1951757  -1.177712
 -1.7175958  -1.3706945 ]

Cosine Similarity ('alice' vs 'rabbit'):
  CBOW Similarity: 0.3985
  Skip-gram Similarity: 0.3717


### Output Explanation: Vector Similarities
- **Dense Representation**: The printed array is a 20-dimensional dense coordinate vector for the word `"alice"`.
- **Similarity Comparison**: The cosine similarity between `"alice"` and `"rabbit"` shows how the embedding projections cluster words that share semantic context.
